# PTCG Imitation-Learning Submission

Attach a trained checkpoint from `training/train.py` and a Dataset containing `cg`. Edit only `MODEL_PATH`, `CG_PATH`, and `DECK` in the parameter cell. The model architecture is restored automatically from the checkpoint. Running all cells creates `/kaggle/working/submission.tar.gz`.

In [ ]:
from pathlib import Path
import torch

# Edit these two paths after attaching your Kaggle Datasets.
MODEL_PATH = Path('/kaggle/input/your-model-dataset/epoch-003.pt')
CG_PATH = Path('/kaggle/input/your-cg-dataset/cg')

# This is the deck used by the submitted agent, not a model-architecture setting.
DECK = [
    5, 5, 13, 19, 19, 19, 19, 66, 66, 140,
    305, 305, 305, 343, 741, 741, 741, 741, 742, 742,
    742, 742, 743, 743, 743, 743, 1079, 1079, 1079, 1081,
    1081, 1081, 1086, 1086, 1086, 1086, 1097, 1129, 1152, 1152,
    1152, 1152, 1182, 1182, 1182, 1184, 1197, 1197, 1197, 1225,
    1225, 1225, 1225, 1231, 1231, 1231, 1231, 1266, 1266, 1266,
]
assert len(DECK) == 60
Path('deck.csv').write_text('\n'.join(map(str, DECK)) + '\n')

assert MODEL_PATH.is_file(), f'Checkpoint not found: {MODEL_PATH}'
assert CG_PATH.is_dir(), f'cg directory not found: {CG_PATH}'
assert (CG_PATH / '__init__.py').is_file(), f'Not a cg package: {CG_PATH}'
try:
    checkpoint_preview = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
except TypeError:
    checkpoint_preview = torch.load(MODEL_PATH, map_location='cpu')
assert isinstance(checkpoint_preview, dict), 'Checkpoint must be a mapping'
assert 'model' in checkpoint_preview and 'config' in checkpoint_preview, (
    'Checkpoint must contain model and config keys'
)
architecture = checkpoint_preview['config']
for key in (
    'd_model', 'd_feedforward', 'num_heads',
    'encoder_layers', 'decoder_layers', 'norm_mode',
    'card_feature_ratio',
):
    print(f'{key}: {architecture[key]}')
del checkpoint_preview
print('Model:', MODEL_PATH)
print('cg:', CG_PATH)
print('Deck cards:', len(DECK))

In [ ]:
%%writefile main.py
from __future__ import annotations

import os
from dataclasses import dataclass, field
from itertools import combinations
from typing import Any

import torch
from cg.api import AreaType, OptionType, all_card_data, to_observation_class

torch.set_num_threads(1)
MAX_ACTIONS = 64
CARD_FEATURE_DIM = 54
CARD_TYPE_DIM = 7
ENERGY_TYPE_DIM = 12
CARD_ENERGY_TYPE_OFFSET = 7
CARD_HP_INDEX = 19
CARD_RETREAT_INDEX = 20
CARD_WEAKNESS_OFFSET = 21
CARD_RESISTANCE_OFFSET = 34
CARD_STAGE_OFFSET = 47
CARD_SPECIAL_OFFSET = 50


def asset_path(name: str) -> str:
    # Kaggle executes main.py with exec(), so __file__ is not guaranteed.
    candidates = [name, os.path.join('/kaggle_simulations/agent', name)]
    module_file = globals().get('__file__')
    if module_file:
        candidates.append(os.path.join(os.path.dirname(module_file), name))
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(name)


def read_deck() -> list[int]:
    with open(asset_path('deck.csv'), encoding='utf-8') as handle:
        deck = [int(line.strip()) for line in handle if line.strip()]
    if len(deck) != 60:
        raise ValueError(f'Expected 60 cards, found {len(deck)}')
    return deck


MY_DECK = read_deck()


def build_card_feature_table(cards, card_count):
    features = torch.zeros((card_count, CARD_FEATURE_DIM), dtype=torch.float32)
    for card in cards:
        card_id = int(card.cardId)
        if not 0 <= card_id < card_count:
            continue
        card_type = int(card.cardType)
        if 0 <= card_type < CARD_TYPE_DIM:
            features[card_id, card_type] = 1
        energy_type = int(card.energyType)
        if 0 <= energy_type < ENERGY_TYPE_DIM:
            features[card_id, CARD_ENERGY_TYPE_OFFSET + energy_type] = 1
        features[card_id, CARD_HP_INDEX] = float(card.hp) / 400
        features[card_id, CARD_RETREAT_INDEX] = float(card.retreatCost) / 5
        weakness = None if card.weakness is None else int(card.weakness)
        weakness = weakness if weakness is not None and 0 <= weakness < ENERGY_TYPE_DIM else ENERGY_TYPE_DIM
        features[card_id, CARD_WEAKNESS_OFFSET + weakness] = 1
        resistance = None if card.resistance is None else int(card.resistance)
        resistance = resistance if resistance is not None and 0 <= resistance < ENERGY_TYPE_DIM else ENERGY_TYPE_DIM
        features[card_id, CARD_RESISTANCE_OFFSET + resistance] = 1
        features[card_id, CARD_STAGE_OFFSET:CARD_SPECIAL_OFFSET] = torch.tensor([
            float(card.basic), float(card.stage1), float(card.stage2),
        ])
        features[card_id, CARD_SPECIAL_OFFSET:CARD_FEATURE_DIM] = torch.tensor([
            float(card.ex), float(card.megaEx), float(card.tera), float(card.aceSpec),
        ])
    return features


@dataclass(frozen=True)
class ModelConfig:
    card_count: int
    attack_count: int
    encoder_size: int = 22_000
    num_encoder_words: int = 24
    decoder_main_features: int = 8
    recover_special_condition: int = 48
    d_model: int = 128
    num_heads: int = 2
    d_feedforward: int = 256
    encoder_layers: int = 1
    decoder_layers: int = 1
    norm_mode: str = 'postnorm'
    card_feature_ratio: float = 0.5

    @property
    def decoder_attack_offset(self) -> int:
        return 14

    @property
    def decoder_card_offset(self) -> int:
        return self.decoder_attack_offset + self.attack_count

    @property
    def decoder_size(self) -> int:
        return self.decoder_card_offset + (
            1 + self.decoder_main_features + self.recover_special_condition
        ) * self.card_count


    @property
    def card_feature_dim(self) -> int:
        return int(self.d_model * self.card_feature_ratio)


def fill_card_range(mapping, start, card_count):
    end = start + card_count
    mapping[start:end] = torch.arange(card_count)
    return end


def encoder_card_ids(config):
    mapping = torch.full((config.encoder_size,), config.card_count, dtype=torch.long)
    position = 0
    for _ in range(4):
        position += 2
        for _ in range(3):
            position = fill_card_range(mapping, position, config.card_count)
    for _ in range(2):
        position += 16
        position = fill_card_range(mapping, position, config.card_count)
    for _ in range(3):
        position = fill_card_range(mapping, position, config.card_count)
    return mapping


def decoder_card_ids(config):
    mapping = torch.full((config.decoder_size,), config.card_count, dtype=torch.long)
    region = torch.arange(config.decoder_size - config.decoder_card_offset)
    mapping[config.decoder_card_offset:] = region % config.card_count
    return mapping


class CardAwareEmbeddingBag(torch.nn.EmbeddingBag):
    def __init__(self, num_embeddings, embedding_dim, index_to_card_id):
        super().__init__(num_embeddings, embedding_dim, mode='sum')
        self.register_buffer('index_to_card_id', index_to_card_id, persistent=False)

    def forward(self, indices, offsets, weights, projected_card_features):
        learned = super().forward(indices, offsets, per_sample_weights=weights)
        static_weights = None if weights is None else weights.to(projected_card_features.dtype)
        static = torch.nn.functional.embedding_bag(
            self.index_to_card_id[indices], projected_card_features, offsets,
            mode='sum', per_sample_weights=static_weights,
        )
        return learned + static


class DecoderLayer(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_feedforward, norm_mode):
        super().__init__()
        self.prenorm = norm_mode == 'prenorm'
        self.attention = torch.nn.MultiheadAttention(d_model, num_heads)
        self.fc1 = torch.nn.Linear(d_model, d_feedforward)
        self.fc2 = torch.nn.Linear(d_feedforward, d_model)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)

    def forward(self, x, encoder_out):
        if self.prenorm:
            query = self.norm1(x)
            attended, _ = self.attention(
                query, encoder_out, encoder_out, need_weights=False
            )
            x = x + attended
            return x + self.fc2(torch.nn.functional.relu(self.fc1(self.norm2(x))))
        y, _ = self.attention(x, encoder_out, encoder_out, need_weights=False)
        residual = self.norm1(x + y)
        y = self.fc2(torch.nn.functional.relu(self.fc1(residual)))
        return self.norm2(residual + y)


class PTCGTransformer(torch.nn.Module):
    def __init__(self, config, card_feature_table):
        super().__init__()
        self.config = config
        self.register_buffer('card_feature_table', card_feature_table)
        self.card_feature_projection = torch.nn.Linear(
            CARD_FEATURE_DIM, config.card_feature_dim
        )
        prenorm = config.norm_mode == 'prenorm'
        self.encoder_bag = CardAwareEmbeddingBag(
            config.encoder_size, config.d_model, encoder_card_ids(config)
        )
        layer = torch.nn.TransformerEncoderLayer(
            config.d_model, config.num_heads, config.d_feedforward,
            dropout=0, norm_first=prenorm
        )
        self.encoder = torch.nn.TransformerEncoder(
            layer, config.encoder_layers,
            norm=torch.nn.LayerNorm(config.d_model) if prenorm else None,
            enable_nested_tensor=False,
        )
        self.decoder_bag = CardAwareEmbeddingBag(
            config.decoder_size, config.d_model, decoder_card_ids(config)
        )
        self.decoder = torch.nn.ModuleList(
            DecoderLayer(
                config.d_model, config.num_heads,
                config.d_feedforward, config.norm_mode,
            )
            for _ in range(config.decoder_layers)
        )
        self.decoder_fc = torch.nn.Linear(config.d_model, 1)

    def project_card_features(self):
        projected = self.card_feature_projection(self.card_feature_table)
        projected = torch.nn.functional.pad(
            projected, (0, self.config.d_model - self.config.card_feature_dim)
        )
        return torch.cat([
            projected, projected.new_zeros((1, self.config.d_model))
        ])

    def forward(self, index_encoder, value_encoder, offset_encoder,
                index_decoder, value_decoder, offset_decoder):
        cfg = self.config
        projected_card_features = self.project_card_features()
        encoded = self.encoder_bag(
            index_encoder, offset_encoder, value_encoder, projected_card_features
        )
        encoded = encoded.reshape(-1, cfg.num_encoder_words, cfg.d_model).transpose(0, 1)
        batch_size = encoded.size(1)
        encoder_out = self.encoder(encoded)
        policy = self.decoder_bag(
            index_decoder, offset_decoder, value_decoder, projected_card_features
        )
        policy = policy.reshape(batch_size, -1, cfg.d_model).transpose(0, 1)
        for layer in self.decoder:
            policy = layer(policy, encoder_out)
        return self.decoder_fc(policy).transpose(0, 1).reshape(batch_size, -1)


@dataclass
class SparseVector:
    index: list[int] = field(default_factory=list)
    value: list[float] = field(default_factory=list)
    offset: list[int] = field(default_factory=list)
    pos: int = 0

    def add(self, index, value):
        if float(value) != 0.0:
            self.index.append(self.pos + int(index))
            self.value.append(float(value))

    def add_pos(self, count):
        self.pos += count

    def add_single(self, value):
        self.add(0, value)
        self.pos += 1

    def word_start(self):
        self.offset.append(len(self.index))


def enumerate_actions(option_count, min_count, max_count):
    actions = []
    for count in range(max_count, min_count - 1, -1):
        for selection in combinations(range(option_count), count):
            actions.append(list(selection))
            if len(actions) == MAX_ACTIONS:
                return actions
    return actions


def add_card(sv, card, card_count):
    if card is not None:
        sv.add(card.id, 1)
    sv.add_pos(card_count)


def add_cards(sv, cards, weight, card_count):
    if cards is not None:
        for card in cards:
            sv.add(card.id, weight)
    sv.add_pos(card_count)


def add_pokemon(sv, pokemon, card_count):
    if pokemon is None:
        sv.add_single(1)
        sv.add_pos(1 + 3 * card_count)
        return
    sv.add_single(0)
    sv.add_single(pokemon.hp / 400)
    add_card(sv, pokemon, card_count)
    add_cards(sv, pokemon.tools, 1, card_count)
    add_cards(sv, pokemon.energyCards, 0.5, card_count)


def add_player(sv, player, card_count):
    for value in (
        player.deckCount / 60, len(player.discard) / 60,
        player.handCount / 8, len(player.bench) / 5,
    ):
        sv.add_single(value)
    sv.add(len(player.prize), 1)
    sv.add_pos(7)
    for value in (
        player.poisoned, player.burned, player.asleep,
        player.paralyzed, player.confused,
    ):
        sv.add_single(value)
    add_cards(sv, player.discard, 0.25, card_count)


def encoder_features(obs, deck, card_count):
    state = obs.current
    yours = state.yourIndex
    sv = SparseVector()
    for i in range(2):
        player = state.players[i ^ yours]
        for j in range(8):
            sv.word_start()
            pos = sv.pos
            add_pokemon(sv, player.bench[j] if j < len(player.bench) else None, card_count)
            if j != 7:
                sv.pos = pos
    for i in range(2):
        player = state.players[i ^ yours]
        sv.word_start()
        add_pokemon(sv, player.active[0] if player.active else None, card_count)
    for i in range(2):
        sv.word_start()
        add_player(sv, state.players[i ^ yours], card_count)
    sv.word_start()
    add_cards(sv, state.players[yours].hand, 0.25, card_count)
    sv.word_start()
    for card_id in deck:
        sv.add(card_id, 0.25)
    sv.add_pos(card_count)
    sv.word_start()
    add_cards(sv, state.stadium, 1, card_count)
    sv.word_start()
    sv.add_single(1)
    sv.add_single(state.turn / 10)
    sv.add_single(state.firstPlayer == yours)
    return sv


def decoder_features(obs, actions, card_count, attack_count):
    sv = SparseVector()
    yours = obs.current.yourIndex
    player_state = obs.current.players[yours]
    context = obs.select.context
    card_offset = 14 + attack_count

    def get_card(option, area=None, index=None, player_index=None):
        area = option.area if area is None else area
        index = option.index if index is None else index
        player_index = yours if player_index is None else player_index
        player = obs.current.players[player_index]
        mapping = {
            AreaType.DECK: obs.select.deck, AreaType.HAND: player.hand,
            AreaType.DISCARD: player.discard, AreaType.ACTIVE: player.active,
            AreaType.BENCH: player.bench, AreaType.PRIZE: player.prize,
            AreaType.STADIUM: obs.current.stadium, AreaType.LOOKING: obs.current.looking,
        }
        cards = mapping.get(area)
        return cards[index] if cards is not None else None

    def main_feature(feature_index, card):
        if card is not None:
            sv.add(card_offset + feature_index * card_count + card.id, 1)

    def context_card(card):
        if card is not None:
            sv.add(card_offset + (8 + int(context)) * card_count + card.id, 1)

    for action in actions:
        sv.word_start()
        if not action:
            sv.add(0, 1)
            continue
        for index in action:
            option = obs.select.option[index]
            if option.type == OptionType.END:
                sv.add(1, 1)
            elif option.type == OptionType.YES:
                sv.add(2, 1)
            elif option.type == OptionType.NO:
                sv.add(3, 1)
            elif option.type == OptionType.SPECIAL_CONDITION:
                sv.add(4 + option.specialConditionType, 1)
            elif option.type == OptionType.NUMBER:
                sv.add(9 + min(option.number, 4), 1)
            elif option.type == OptionType.ATTACK:
                sv.add(14 + option.attackId, 1)
            elif option.type == OptionType.PLAY:
                main_feature(0, player_state.hand[option.index])
            elif option.type == OptionType.ATTACH:
                main_feature(1, get_card(option))
                main_feature(2, get_card(option, option.inPlayArea, option.inPlayIndex))
            elif option.type == OptionType.EVOLVE:
                main_feature(3, get_card(option))
                main_feature(4, get_card(option, option.inPlayArea, option.inPlayIndex))
            elif option.type == OptionType.ABILITY:
                main_feature(5, get_card(option))
            elif option.type == OptionType.DISCARD:
                main_feature(6, get_card(option))
            elif option.type == OptionType.RETREAT:
                main_feature(7, player_state.active[0])
            elif option.type == OptionType.CARD:
                context_card(get_card(option, player_index=option.playerIndex))
            elif option.type == OptionType.TOOL_CARD:
                card = get_card(option, player_index=option.playerIndex)
                context_card(card.tools[option.toolIndex])
            elif option.type in (OptionType.ENERGY_CARD, OptionType.ENERGY):
                card = get_card(option, player_index=option.playerIndex)
                context_card(card.energyCards[option.energyIndex])
            elif option.type == OptionType.SKILL:
                sv.add(card_offset + (8 + int(context)) * card_count + option.cardId, 1)
    return sv


def sparse_tensors(vector):
    return (
        torch.tensor(vector.index, dtype=torch.int64),
        torch.tensor(vector.value, dtype=torch.float32),
        torch.tensor(vector.offset, dtype=torch.int64),
    )


def load_model():
    path = asset_path('model.pt')
    try:
        checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        checkpoint = torch.load(path, map_location='cpu')
    if not isinstance(checkpoint, dict) or 'model' not in checkpoint or 'config' not in checkpoint:
        raise ValueError('Expected checkpoint with model and config keys')
    config = ModelConfig(**checkpoint['config'])
    card_feature_table = build_card_feature_table(all_card_data(), config.card_count)
    model = PTCGTransformer(config, card_feature_table)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    return model, config


MODEL, CONFIG = load_model()


def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return MY_DECK

    actions = enumerate_actions(
        len(obs.select.option), obs.select.minCount, obs.select.maxCount
    )
    if not actions:
        return list(range(min(obs.select.maxCount, len(obs.select.option))))
    encoder = encoder_features(obs, MY_DECK, CONFIG.card_count)
    decoder = decoder_features(obs, actions, CONFIG.card_count, CONFIG.attack_count)
    with torch.inference_mode():
        policy_logits = MODEL(*sparse_tensors(encoder), *sparse_tensors(decoder))
    return actions[int(policy_logits[0].argmax().item())]


In [ ]:
# Syntax-check the generated agent without importing it yet.
compile(Path('main.py').read_text(), 'main.py', 'exec')
print('main.py syntax OK')

In [ ]:
import tarfile

with tarfile.open('submission.tar.gz', 'w:gz') as tar:
    tar.add('main.py', arcname='main.py')
    tar.add('deck.csv', arcname='deck.csv')
    tar.add(MODEL_PATH, arcname='model.pt')
    tar.add(CG_PATH, arcname='cg')

size_mb = Path('submission.tar.gz').stat().st_size / 1024**2
print(f'Created submission.tar.gz ({size_mb:.2f} MB)')
with tarfile.open('submission.tar.gz', 'r:gz') as tar:
    print('Archive root:', sorted({name.split('/')[0] for name in tar.getnames()}))